In [1]:
# ──────────────────────────────────────────────────────────────────────
# Cell 1 · Dependencies — transformers PINNED to Baidu's tested version
# ──────────────────────────────────────────────────────────────────────
!pip install -q "transformers==4.57.1" accelerate einops addict easydict \
    pymupdf psutil gradio safetensors sentencepiece

import transformers
print("transformers version:", transformers.__version__)   # must print 4.57.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers version: 4.57.1


In [2]:
# ──────────────────────────────────────────────────────────────────────
# Cell 2 · Defensive compat layer + log silencing
#   Every patch below is guarded: on 4.57.1 most are no-ops.
# ──────────────────────────────────────────────────────────────────────
import warnings
import logging

def apply_compat_patches() -> None:
    # 0) Silence hub/transformers noise
    for lg in ["transformers", "transformers.modeling_utils",
               "transformers.models.auto.auto_factory",
               "huggingface_hub", "huggingface_hub.utils._http"]:
        logging.getLogger(lg).setLevel(logging.ERROR)
    warnings.filterwarnings("ignore")

    # 1) Utility flags (exist on 4.57 → skipped automatically)
    import transformers.utils.import_utils as import_utils
    import transformers.utils as utils
    for name in ("is_torch_fx_available", "is_flash_attn_2_available",
                 "is_flash_attn_available", "is_torch_sdpa_available"):
        for module in (import_utils, utils):
            if not hasattr(module, name):
                setattr(module, name, lambda: False)

    # 2) Flash-attn class stub (only if removed)
    from transformers.models.llama import modeling_llama
    if not hasattr(modeling_llama, "LlamaFlashAttention2"):
        class LlamaFlashAttention2(modeling_llama.LlamaAttention):
            """Compatibility stub."""
        modeling_llama.LlamaFlashAttention2 = LlamaFlashAttention2

    # 3) SentencePiece slow-tokenizer guard (repo ships tokenizer.json only)
    import sentencepiece as spm
    _orig_load = spm.SentencePieceProcessor.Load
    def _safe_load(self, model_file=None, model_proto=None):
        if model_file is None and model_proto is None:
            return
        return _orig_load(self, model_file, model_proto)
    spm.SentencePieceProcessor.Load = _safe_load

    # 4) bf16 → fp16 downgrade on GPUs without bf16 (autocast + Tensor.to)
    import torch
    _orig_autocast = torch.autocast
    def _safe_autocast(device_type="cuda", dtype=None, *args, **kwargs):
        if (dtype == torch.bfloat16 and str(device_type).startswith("cuda")
                and not torch.cuda.is_bf16_supported()):
            dtype = torch.float16
        return _orig_autocast(device_type, dtype=dtype, *args, **kwargs)
    torch.autocast = _safe_autocast

    _orig_to = torch.Tensor.to
    def _safe_to(self, *args, **kwargs):
        if not torch.cuda.is_bf16_supported():
            args = tuple(torch.float16 if a is torch.bfloat16 else a for a in args)
            if kwargs.get("dtype", None) is torch.bfloat16:
                kwargs["dtype"] = torch.float16
        return _orig_to(self, *args, **kwargs)
    torch.Tensor.to = _safe_to

    # 5) Defensive attention-mask length aligner (no-op unless mismatched)
    import transformers.modeling_attn_mask_utils as attn_utils
    _orig_prep = attn_utils._prepare_4d_causal_attention_mask
    def _prep_4d(attention_mask, input_shape, inputs_embeds,
                 past_key_values_length, sliding_window=None, **kw):
        if inputs_embeds is not None:
            tgt = inputs_embeds.shape[1]
            if isinstance(input_shape, tuple) and len(input_shape) >= 2 \
                    and input_shape[1] != tgt:
                input_shape = (input_shape[0], tgt)
            if (attention_mask is not None and hasattr(attention_mask, "dim")
                    and attention_mask.dim() == 2
                    and attention_mask.shape[-1] != tgt):
                b = attention_mask.shape[0]
                nm = torch.ones((b, tgt), dtype=attention_mask.dtype,
                                device=attention_mask.device)
                ov = min(attention_mask.shape[-1], tgt)
                nm[:, :ov] = attention_mask[:, :ov]
                attention_mask = nm
        if sliding_window is not None:
            return _orig_prep(attention_mask, input_shape, inputs_embeds,
                              past_key_values_length,
                              sliding_window=sliding_window, **kw)
        return _orig_prep(attention_mask, input_shape, inputs_embeds,
                          past_key_values_length, **kw)
    attn_utils._prepare_4d_causal_attention_mask = _prep_4d

apply_compat_patches()
print("✅ Compat layer applied (4.57-safe).")

✅ Compat layer applied (4.57-safe).


In [3]:
# ──────────────────────────────────────────────────────────────────────
# Cell 3 · Config + model load
#   NOTE: we NO LONGER replace ATTENTION_CLASSES. Baidu's own attention
#   class is correct for 4.57.1; swapping it was the v5 workaround that
#   caused the position_embeddings crash.
# ──────────────────────────────────────────────────────────────────────
import os, gc, sys, torch

gc.collect(); torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

try:
    from google.colab import userdata
    from huggingface_hub import login
    _t = userdata.get('HF_TOKEN')
    if _t: login(token=_t)
except Exception:
    pass

CONFIG = {
    "model_name": "baidu/Unlimited-OCR",
    "dpi": 200,
    "chunk_pages": 1,
    "image_size": 1024,
    "max_length": 16384,
    "no_repeat_ngram_size": 35,
    "ngram_window": 1024,
    "demo_user": "client",
    "demo_pass": "demo123",
}

def load_ocr_model(cfg: dict):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    name = cfg["model_name"]
    tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True, use_fast=True)
    config = AutoConfig.from_pretrained(name, trust_remote_code=True)

    # Guard 1: fill any missing DeepseekV2 defaults (no-op if config.json has them)
    _D = {
        "moe_intermediate_size": 1407, "n_shared_experts": None, "n_routed_experts": None,
        "ep_size": 1, "routed_scaling_factor": 1.0, "kv_lora_rank": 512,
        "q_lora_rank": 1536, "qk_rope_head_dim": 64, "v_head_dim": 128,
        "qk_nope_head_dim": 128, "topk_method": 'gready', "n_group": None,
        "topk_group": None, "num_experts_per_tok": None, "moe_layer_freq": 1,
        "first_k_dense_replace": 0, "norm_topk_prob": False, "scoring_func": 'softmax',
        "aux_loss_alpha": 0.001, "seq_aux": True,
        "pad_token_id": None, "bos_token_id": 100000, "eos_token_id": 100001,
        "attention_bias": False, "attention_dropout": 0.0, "mlp_bias": False,
        "sliding_window": None, "use_mla": True,
    }
    for a, d in _D.items():
        if not hasattr(config, a):
            object.__setattr__(config, a, d)

    # Guard 2: rope_parameters dict (only if absent/None)
    if getattr(config, "rope_parameters", None) is None:
        object.__setattr__(config, "rope_parameters", {
            "rope_type": "default",
            "rope_theta": getattr(config, "rope_theta", 10000.0)})

    # Guard 3: integer dims
    hs = getattr(config, "hidden_size", 4096); nh = getattr(config, "num_attention_heads", 32)
    if getattr(config, "head_dim", None) is None:
        object.__setattr__(config, "head_dim", hs // nh)
    if getattr(config, "num_key_value_heads", None) is None:
        object.__setattr__(config, "num_key_value_heads", nh)

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"GPU bf16 support: {torch.cuda.is_bf16_supported()} → loading weights in {dtype}")

    model = AutoModel.from_pretrained(
        name, config=config, trust_remote_code=True,
        use_safetensors=True, torch_dtype=dtype, low_cpu_mem_usage=True)

    # Guard 4: GenerationMixin (auto-added on 4.57; belt-and-braces)
    from transformers import GenerationMixin
    if not isinstance(model, GenerationMixin):
        model.__class__ = type(model.__class__.__name__, (model.__class__, GenerationMixin), {})

    # Guard 5: vision position-ids / table realignment (defensive)
    try:
        emb = model.model.vision_model.embeddings
        rows = emb.position_embedding.weight.shape[0]
        ids = emb.position_ids
        if ids.numel() == 0 or ids.numel() != rows or int(ids.max()) >= rows or int(ids.min()) < 0:
            nid = torch.arange(rows, dtype=torch.long, device=ids.device)
            nid = nid.view(1, -1) if ids.dim() == 2 else nid
            emb.position_ids = (torch.nn.Parameter(nid, requires_grad=False)
                                if isinstance(emb.position_ids, torch.nn.Parameter) else nid)
    except Exception:
        pass

    model = model.eval().cuda()
    gc.collect(); torch.cuda.empty_cache()
    return tokenizer, model

TOKENIZER, MODEL = load_ocr_model(CONFIG)
print("✅ Model loaded on GPU.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

modeling_unlimitedocr.py: 0.00B [00:00, ?B/s]

...

modeling_deepseekv2.py: 0.00B [00:00, ?B/s]

GPU bf16 support: True → loading weights in torch.bfloat16


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

✅ Model loaded on GPU.


In [7]:
# ──────────────────────────────────────────────────────────────────────
# DIAGNOSTIC · pinpoint the asserting kernel (run after Cell 3)
# ──────────────────────────────────────────────────────────────────────
import os, glob, shutil, tempfile, traceback
from PIL import Image, ImageDraw

# ── Sanity check #1: embedding table vs tokenizer vocab ──
vocab_cfg = MODEL.config.vocab_size
vocab_tok = len(TOKENIZER)
max_special = max(TOKENIZER.all_special_ids, default=-1)
print(f"config.vocab_size={vocab_cfg} | len(tokenizer)={vocab_tok} | max_special_id={max_special}")
if vocab_tok > vocab_cfg or max_special >= vocab_cfg:
    print("❌ MISMATCH: tokenizer can emit ids the embedding table cannot hold → this IS the assert.")
else:
    print("✅ vocab sizes consistent.")

# ── Sanity check #2: vision config ──
print("use_mla:", getattr(MODEL.config, "use_mla", None),
      "| image_size cfg:", CONFIG["image_size"],
      "| head_dim:", getattr(MODEL.config, "head_dim", None))

# ── Minimal repro image ──
tmp = tempfile.mkdtemp(prefix="ocr_diag_")
img_path = os.path.join(tmp, "test.png")
img = Image.new("RGB", (1024, 1024), "white")
d = ImageDraw.Draw(img)
d.text((80, 200), "UNLIMITED OCR DIAGNOSTIC TEST", fill="black")
d.text((80, 400), "The quick brown fox jumps over the lazy dog.", fill="black")
d.text((80, 600), "Numbers: 0123456789 - End of test page.", fill="black")
img.save(img_path)

def try_infer(tag, **kw):
    out = tempfile.mkdtemp(prefix="ocr_diag_out_")
    print(f"\n===== {tag} =====")
    try:
        res = MODEL.infer(TOKENIZER, prompt="<image>document parsing.",
                          image_file=img_path, output_path=out, save_results=True, **kw)
        print("OK →", repr(res)[:300])
    except Exception:
        traceback.print_exc()
    finally:
        shutil.rmtree(out, ignore_errors=True)

# Config A: "base" mode (what infer_multi uses internally)
try_infer("base mode  (image_size=1024, crop_mode=False)",
          base_size=1024, image_size=1024, crop_mode=False, max_length=4096)

# Config B: "gundam" mode (documented default for single images)
try_infer("gundam mode (base_size=1024, image_size=640, crop_mode=True)",
          base_size=1024, image_size=640, crop_mode=True, max_length=4096)

shutil.rmtree(tmp, ignore_errors=True)
print("\n✅ Diagnostic done — paste the FULL traceback of whichever config asserted.")

config.vocab_size=129280 | len(tokenizer)=128827 | max_special_id=128826
✅ vocab sizes consistent.
use_mla: False | image_size cfg: 1024 | head_dim: 128

===== base mode  (image_size=1024, crop_mode=False) =====
<|det|>text [76, 193, 241, 206]<|/det|>UNLIMITEDOCR DIAGNOSTIC TEST
<|det|>text [76, 390, 281, 406]<|/det|>The quick brown fox jumps over the lazy dog.
<|det|>text [76, 585, 263, 599]<|/det|>Numbers 0123456789 - End of test page.
===============save results:===============


image: 0it [00:00, ?it/s]
other: 100%|██████████| 3/3 [00:00<00:00, 42366.71it/s]

OK → None

===== gundam mode (base_size=1024, image_size=640, crop_mode=True) =====


<|det|>text [76, 194, 241, 209]<|/det|>UNLIMITEDOCR DIAGNOSTIC TEST
<|det|>text [76, 389, 283, 406]<|/det|>The quick brown fox jumps over the lazy dog.
<|det|>text [75, 584, 263, 600]<|/det|>Numbers 0123456789 - End of test page.
===============save results:===============


image: 0it [00:00, ?it/s]
other: 100%|██████████| 3/3 [00:00<00:00, 40201.00it/s]

OK → None

✅ Diagnostic done — paste the FULL traceback of whichever config asserted.


In [4]:
# ──────────────────────────────────────────────────────────────────────
# Cell 4 · PDF rendering + OCR output cleanup
# ──────────────────────────────────────────────────────────────────────
import os, re, shutil, tempfile, unicodedata
from pathlib import Path
import pymupdf

DET_TAG_RE = re.compile(r"<\|det\|>([^<\s]+)(?:\s*\[[^\]]*\])?\s*<\|/det\|>(.*)", re.DOTALL)
PAGE_SPLIT_RE = re.compile(r"(?:<PAGE>\s*)|(?:<\|page\|>\s*)|(?:\f+)", re.IGNORECASE)

def normalize_text(text: str) -> str:
    if not text: return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = "\n".join(l.rstrip() for l in text.splitlines())
    return re.sub(r"\n{4,}", "\n\n\n", text).strip()

def clean_page(raw_page: str) -> str:
    if not raw_page: return ""
    text = re.sub(r"<\|[^|]*\|>", "", str(raw_page))
    m = DET_TAG_RE.search(text)
    if m: text = m.group(2)
    return normalize_text(text)

def split_model_output(raw: str, expected_pages: int = 1) -> list:
    if raw is None: return [""] * max(expected_pages, 1)
    raw = str(raw).strip()
    if not raw: return [""] * max(expected_pages, 1)
    parts = [p.strip() for p in PAGE_SPLIT_RE.split(raw) if p.strip()]
    if not parts: parts = [raw]
    pages = [clean_page(p) for p in parts if clean_page(p)]
    if expected_pages == 1:
        return ["\n\n".join(pages).strip()]
    if expected_pages and len(pages) > expected_pages:
        pages = pages[:expected_pages - 1] + ["\n\n".join(pages[expected_pages - 1:]).strip()]
    if expected_pages and len(pages) < expected_pages:
        pages.extend([""] * (expected_pages - len(pages)))
    return pages

def render_pdf_pages(pdf_path: str, dpi: int = 200):
    tmp = tempfile.mkdtemp(prefix="unlimited_ocr_pdf_")
    paths = []
    doc = pymupdf.open(str(pdf_path))
    try:
        mat = pymupdf.Matrix(dpi / 72.0, dpi / 72.0)
        for i in range(len(doc)):
            p = os.path.join(tmp, f"page_{i + 1:04d}.png")
            doc[i].get_pixmap(matrix=mat, alpha=False).save(p)
            paths.append(p)
    finally:
        doc.close()
    return paths, tmp

print("✅ Cell 4 loaded.")

✅ Cell 4 loaded.


In [5]:
# ──────────────────────────────────────────────────────────────────────
# Cell 5 · Inference engine (hardened, self-diagnosing)
# ──────────────────────────────────────────────────────────────────────
import gc, glob, os, shutil, tempfile, time
from pathlib import Path
import torch

def free_gpu() -> None:
    gc.collect(); torch.cuda.empty_cache()

def _read_saved_ocr_text(out_dir: str) -> str:
    d = Path(out_dir)
    if not d.exists(): return ""
    fs = list(d.rglob("*.md")) + list(d.rglob("*.txt"))
    return "\n\n".join(f.read_text(encoding="utf-8", errors="ignore")
                       for f in sorted(fs) if f.is_file()).strip()

def _extract_text(result, out_dir: str) -> str:
    if result is None: return _read_saved_ocr_text(out_dir)
    if isinstance(result, str): return result
    if isinstance(result, (tuple, list)):
        ss = [i for i in result if isinstance(i, str) and i.strip()]
        if len(ss) == len(result) and len(result) > 1:
            return "\n\n<PAGE>\n\n".join(ss)
        if ss: return ss[0]
        return _read_saved_ocr_text(out_dir)
    return str(result)

def ocr_image_batch(model, tokenizer, image_paths: list, cfg: dict) -> str:
    out = tempfile.mkdtemp(prefix="unlimited_ocr_out_")
    try:
        common = dict(prompt="<image>Multi page parsing.", image_files=image_paths,
                      output_path=out, image_size=cfg.get("image_size", 1024),
                      max_length=cfg.get("max_length", 32768), save_results=True)
        try:
            res = model.infer_multi(tokenizer,
                                    no_repeat_ngram_size=cfg.get("no_repeat_ngram_size", 35),
                                    ngram_window=cfg.get("ngram_window", 1024), **common)
        except TypeError as e:
            print(f"⚠️ kwarg mismatch ({e}); minimal retry.")
            res = model.infer_multi(tokenizer, **common)
        raw = _extract_text(res, out)
        if not raw.strip():
            print(f"[debug] empty: type={type(res).__name__} repr={repr(res)[:300]}")
        else:
            print(f"[debug] OK raw_len={len(raw)}")
        return raw.strip()
    finally:
        shutil.rmtree(out, ignore_errors=True)

def ocr_pages_safe(model, tokenizer, image_paths: list, cfg: dict) -> list:
    n = len(image_paths)
    if n == 0: return []
    try:
        raw = ocr_image_batch(model, tokenizer, image_paths, cfg)
        pages = split_model_output(raw, expected_pages=n)
        if len(pages) == n: return pages
        print(f"⚠️ chunk {len(pages)}/{n}; retrying 1-by-1.")
    except torch.cuda.OutOfMemoryError:
        print("⚠️ OOM chunk; retrying 1-by-1."); free_gpu()
    except Exception as e:
        print(f"⚠️ chunk failed {type(e).__name__}: {e}; retrying 1-by-1.")
    texts = []
    for i, p in enumerate(image_paths, 1):
        try:
            raw = ocr_image_batch(model, tokenizer, [p], cfg)
            pg = split_model_output(raw, expected_pages=1)
            texts.append(pg[0] if pg else "")
        except torch.cuda.OutOfMemoryError:
            print(f"❌ OOM page {i}."); free_gpu(); texts.append("")
        except Exception as e:
            print(f"❌ page {i} failed {type(e).__name__}: {e}"); texts.append("")
        finally:
            free_gpu()
    return texts

def ocr_pdf_streaming(model, tokenizer, pdf_path: str, cfg: dict, max_pages=None):
    free_gpu()
    paths, tmp = [], None
    try:
        paths, tmp = render_pdf_pages(pdf_path, dpi=cfg.get("dpi", 200))
        if max_pages is not None: paths = paths[:max_pages]
        total = len(paths)
        if total == 0:
            yield "⚠️ No pages found.", [], 0, 0; return
        chunk = max(1, int(cfg.get("chunk_pages", 1)))
        texts, t0 = [], time.time()
        for i in range(0, total, chunk):
            b = paths[i:i + chunk]; lo, hi = i + 1, i + len(b)
            yield (f"🧠 OCR page(s) {lo}–{hi} of {total}...", list(texts), i, total)
            texts.extend(ocr_pages_safe(model, tokenizer, b, cfg))
            yield (f"✅ page(s) {lo}–{hi} done ({time.time() - t0:.0f}s).",
                   list(texts), hi, total)
        if not any(t.strip() for t in texts):
            yield ("❌ EMPTY text on all pages — see console [debug] lines.",
                   list(texts), total, total)
        else:
            yield (f"🎉 OCR complete. {total} page(s).", list(texts), total, total)
    finally:
        if tmp: shutil.rmtree(tmp, ignore_errors=True)
        free_gpu()

print("✅ Cell 5 loaded.")

✅ Cell 5 loaded.


In [6]:
# ──────────────────────────────────────────────────────────────────────
# Cell 6 · Gradio demo UI
# ──────────────────────────────────────────────────────────────────────
import os, torch, gradio as gr

PAGE_BREAK = "\n\n--- Page Break ---\n\n"
DEMO_MAX_FILE_SIZE_MB = 1     # set None for production

def format_pages_for_display(page_texts: list) -> str:
    out = []
    for i, t in enumerate(page_texts, 1):
        c = clean_page(t)
        out.append(f"## Page {i}\n\n{c}" if c else f"## Page {i}\n\n(no text detected)")
    return PAGE_BREAK.join(out).strip()

def demo_handler(pdf_file, progress=gr.Progress()):
    if pdf_file is None:
        yield "⚠️ Please upload a PDF file.", ""; return
    pdf_path = pdf_file if isinstance(pdf_file, str) else pdf_file.name
    if not str(pdf_path).lower().endswith(".pdf"):
        yield "⚠️ Please upload a valid PDF file.", ""; return
    if DEMO_MAX_FILE_SIZE_MB is not None:
        mb = os.path.getsize(pdf_path) / (1024 * 1024)
        if mb > DEMO_MAX_FILE_SIZE_MB:
            yield (f"⚠️ File is {mb:.2f} MB; demo limit {DEMO_MAX_FILE_SIZE_MB} MB.", ""); return
    latest = ""
    try:
        for msg, pages, done, total in ocr_pdf_streaming(MODEL, TOKENIZER, pdf_path, CONFIG):
            if total: progress(done / max(total, 1), desc=msg)
            latest = format_pages_for_display(pages)
            yield msg, latest
    except torch.cuda.OutOfMemoryError:
        free_gpu()
        yield ("❌ CUDA OOM. Lower CONFIG['image_size'] to 768 or dpi to 150.", latest)
    except Exception as e:
        yield f"❌ Failed: {type(e).__name__}: {e}", latest

def client_auth(u, p):
    return u == CONFIG.get("demo_user", "client") and p == CONFIG.get("demo_pass", "demo123")

def build_demo_app():
    with gr.Blocks(title="Unlimited-OCR PDF Demo") as app:
        gr.Markdown("# Unlimited-OCR PDF Extraction Demo\n\nUpload a PDF to extract text.")
        with gr.Row():
            fin = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
        btn = gr.Button("🚀 Run OCR", variant="primary")
        st = gr.Textbox(label="Status", lines=3, interactive=False)
        tx = gr.Textbox(label="Extracted Text", lines=28, interactive=False)
        btn.click(fn=demo_handler, inputs=[fin], outputs=[st, tx])
    return app

APP = build_demo_app()
APP.launch(share=True, auth=client_auth)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c9c0e3bc410a360d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
